In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
import warnings

warnings.filterwarnings('ignore')

df = pd.read_csv('Breast Cancer METABRIC.csv')

target_col = 'Tumor Stage'
original_count = len(df)
df = df[df[target_col] != 4].copy()
removed_count = original_count - len(df)
df = df.dropna(subset=[target_col])
leakage_features = [
    'Overall Survival (Months)',
    'Overall Survival Status',
    'Patient\'s Vital Status',
    'Relapse Free Status (Months)',
    'Relapse Free Status',
    'Nottingham prognostic index',
]
removed_leakage = [f for f in leakage_features if f in df.columns]
df = df.drop(columns=removed_leakage)

missing_rates = df.isnull().sum() / len(df)
high_missing = missing_rates[missing_rates > 0.8].index.tolist()
high_missing = [c for c in high_missing if c != target_col]
if high_missing:
    df = df.drop(columns=high_missing)
    print(f"{len(high_missing)} features with high missing rate have been removed")

exclude_kw = ['stage', 'unnamed', 'patient id', 'patient\'s vital',
              'survival', 'relapse', 'status']

feature_cols = []
for col in df.columns:
    if col == target_col:
        continue
    if any(kw in col.lower() for kw in exclude_kw):
        continue
    if df[col].isnull().sum() / len(df) < 0.5:
        feature_cols.append(col)

X = df[feature_cols].copy()
y = df[target_col].copy()

# Handle missing values
for col in X.select_dtypes(include=[np.number]).columns:
    if X[col].isnull().any():
        X[col].fillna(X[col].median(), inplace=True)

for col in X.select_dtypes(include=['object']).columns:
    if X[col].isnull().any():
        mode = X[col].mode()[0] if len(X[col].mode()) > 0 else 'Unknown'
        X[col].fillna(mode, inplace=True)

# Label encoding
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

print(f"Feature matrix: {X.shape}")

# Feature selection
K_FEATURES = 15

selector = SelectKBest(score_func=mutual_info_classif, k=min(K_FEATURES, len(feature_cols)))
selector.fit(X, y)

feature_scores = pd.DataFrame({
    'feature': feature_cols,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print(feature_scores.head(20).to_string(index=False))

selected_mask = selector.get_support()
selected_features = [feature_cols[i] for i in range(len(feature_cols)) if selected_mask[i]]

for i, feat in enumerate(selected_features, 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:<65} (Score: {score:.4f})")

X_selected = X[selected_features].copy()

# Data splitting
class_counts = y.value_counts()
print("\nNumber of samples per stage:")
for stage, count in class_counts.items():
    print(f"  Stage {stage}: {count} cases")

min_samples = class_counts.min()
stratify_param = None if min_samples < 2 else y

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.3,
    random_state=42,
    stratify=stratify_param
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining set distribution (before SMOTE):")
for s, c in y_train.value_counts().sort_index().items():
    print(f"  Stage {s}: {c} cases")

# SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nBefore SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_bal.shape}")
print(f"Added samples: {X_train_bal.shape[0] - X_train.shape[0]}")

print("\nTraining set distribution (after SMOTE):")
for s, c in pd.Series(y_train_bal).value_counts().sort_index().items():
    print(f"  Stage {s}: {c} cases")


le_y = LabelEncoder()
y_train_encoded = le_y.fit_transform(y_train_bal)
y_test_encoded = le_y.transform(y_test)
y_train_original_encoded = le_y.transform(y_train)

num_classes = len(le_y.classes_)
print(f"\nNumber of classes: {num_classes}")
print(f"Class labels: {le_y.classes_}")

y_train_categorical = to_categorical(y_train_encoded, num_classes)
y_test_categorical = to_categorical(y_test_encoded, num_classes)


from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)
X_train_original_scaled = scaler.transform(X_train)


mlnet_config = {
    'n_dense_layers': 3,
    'dense_neurons': [256, 128, 64],
    'dropout_rate': 0.5,
    'learning_rate': 0.0005,
}

print("\nOptimized MLNet architecture parameters:")
print(f"  Number of fully connected layers: {mlnet_config['n_dense_layers']}")
print(f"  Neurons per layer: {mlnet_config['dense_neurons']}")
print(f"  Dropout rate: {mlnet_config['dropout_rate']}")
print(f"  Learning rate: {mlnet_config['learning_rate']}")


def build_mlnet(input_dim, num_classes, config):
    # MLNet model based on PSO+DHDE optimized architecture from paper
    model = models.Sequential()

    # Input layer
    model.add(layers.Input(shape=(input_dim,)))


    for i, neurons in enumerate(config['dense_neurons']):
        model.add(layers.Dense(
            neurons,
            activation='relu',
            kernel_initializer='he_normal',
            name=f'dense_{i+1}'
        ))
        model.add(layers.Dropout(config['dropout_rate'], name=f'dropout_{i+1}'))

        model.add(layers.BatchNormalization(name=f'bn_{i+1}'))

    model.add(layers.Dense(num_classes, activation='softmax', name='output'))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


mlnet_model = build_mlnet(
    input_dim=X_train_scaled.shape[1],
    num_classes=num_classes,
    config=mlnet_config
)

mlnet_model.summary()

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

print(f"\nTraining samples: {X_train_scaled.shape[0]}")
print(f"Validation split: 15%")
print(f"Batch size: 32")
print(f"Max epochs: 200")

history = mlnet_model.fit(
    X_train_scaled, y_train_categorical,
    epochs=200,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stopping],
    verbose=1
)

# Cross-validation
from sklearn.model_selection import StratifiedKFold

X_full_scaled = scaler.fit_transform(X_selected)
y_full_encoded = le_y.fit_transform(y)
y_full_categorical = to_categorical(y_full_encoded, num_classes)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

print("\n5-fold cross-validation:")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X_full_scaled, y_full_encoded), 1):
    fold_model = build_mlnet(
        input_dim=X_full_scaled.shape[1],
        num_classes=num_classes,
        config=mlnet_config
    )

    fold_model.fit(
        X_full_scaled[train_idx], y_full_categorical[train_idx],
        epochs=100,
        batch_size=32,
        verbose=0,
        validation_split=0.15,
        callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)]
    )

    _, accuracy = fold_model.evaluate(X_full_scaled[val_idx], y_full_categorical[val_idx], verbose=0)
    cv_scores.append(accuracy)
    print(f"  Fold {fold}: {accuracy:.4f}")

cv_scores = np.array(cv_scores)
print(f"\naverage: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Model evaluation


y_train_pred_proba = mlnet_model.predict(X_train_original_scaled, verbose=0)
y_train_pred = np.argmax(y_train_pred_proba, axis=1)
y_train_true = le_y.transform(y_train)

y_test_pred_proba = mlnet_model.predict(X_test_scaled, verbose=0)
y_test_pred = np.argmax(y_test_pred_proba, axis=1)
y_test_true = le_y.transform(y_test)

train_accuracy = accuracy_score(y_train_true, y_train_pred)
train_precision = precision_score(y_train_true, y_train_pred, average='weighted', zero_division=0)
train_recall = recall_score(y_train_true, y_train_pred, average='weighted', zero_division=0)
train_f1 = f1_score(y_train_true, y_train_pred, average='weighted', zero_division=0)

test_accuracy = accuracy_score(y_test_true, y_test_pred)
test_precision = precision_score(y_test_true, y_test_pred, average='weighted', zero_division=0)
test_recall = recall_score(y_test_true, y_test_pred, average='weighted', zero_division=0)
test_f1 = f1_score(y_test_true, y_test_pred, average='weighted', zero_division=0)

print("=" * 70)
print(" " * 20 + "Model Performance Comparison")
print("=" * 70)
print(f"{'Metric':<15} {'Training Set':>15} {'Test Set':>15} {'Difference':>15}")
print("-" * 70)
print(f"{'Accuracy':<15} {train_accuracy:>15.4f} {test_accuracy:>15.4f} {abs(train_accuracy-test_accuracy):>15.4f}")
print(f"{'Precision':<15} {train_precision:>15.4f} {test_precision:>15.4f} {abs(train_precision-test_precision):>15.4f}")
print(f"{'Recall':<15} {train_recall:>15.4f} {test_recall:>15.4f} {abs(train_recall-test_recall):>15.4f}")
print(f"{'F1 Score':<15} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print("=" * 70)

print(classification_report(
    y_test_true, y_test_pred,
    target_names=[str(c) for c in le_y.classes_],
    zero_division=0
))

